# Predict early-season FPL performance from the previous season aggregates


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from prediction.artifacts.io import save_trained_catboost_model
from prediction.artifacts.path_registry import PRE_SEASON_ARTIFACT_PATH
from training.load_training_data import load_historic_player_fixture_data
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
EARLY_GAMEWEEKS = 10

# Training and evaluation

For training use aggregate values from the 2021/22 and 2022/23 seasons to predict the performance of the same player in the early gameweeks of the following season

## Load feature and target seasons


In [ ]:
TRAINING_SEASONS = ["2021-22", "2022-23", "2023-24"]

fixture_history_df = pd.concat(
    [
        load_historic_player_fixture_data(season).assign(season=season)
        for season in TRAINING_SEASONS
    ],
    ignore_index=True,
    sort=False,
)

print(fixture_history_df.groupby("season").size())
fixture_history_df.head()

# Feature list

In [ ]:
CATEGORICAL_COLUMNS = ["position"]

NUMERIC_FEATURES = [
    "total_points",
    "minutes",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "own_goals",
    "penalties_saved",
    "penalties_missed",
    "yellow_cards",
    "red_cards",
    "saves",
    "bonus",
    "bps",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "starts",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
]

TARGET_COLUMN = "target_avg_fixture_points_first_gws"

# Create training dataset

In [ ]:
fixture_history_df = fixture_history_df.sort_values(
    ["season", "name", "GW", "fixture"]
).copy()
fixture_history_df[NUMERIC_FEATURES] = fixture_history_df[
    NUMERIC_FEATURES
].apply(
    pd.to_numeric, errors="coerce"
)

player_season_categories = (
    fixture_history_df.groupby(["season", "name"], as_index=False)[
        CATEGORICAL_COLUMNS
    ].last()
)

player_season_totals = (
    fixture_history_df.groupby(["season", "name"], as_index=False)[
        NUMERIC_FEATURES
    ]
    .sum(min_count=1)
    .rename(columns={
        feature: f"season_sum_{feature}" for feature in NUMERIC_FEATURES
    })
)
player_season_features = player_season_categories.merge(
    player_season_totals,
    on=["season", "name"],
    how="inner",
    validate="one_to_one",
)

early_fixture_history = fixture_history_df.loc[
    fixture_history_df["GW"].between(1, EARLY_GAMEWEEKS)
].copy()

early_fixture_targets = (
    early_fixture_history.groupby(["season", "name"], as_index=False)
    .agg(**{TARGET_COLUMN: ("total_points", "mean")})
    .rename(columns={"season": "target_season"})
)

subsequent_season = dict(zip(TRAINING_SEASONS, TRAINING_SEASONS[1:]))
player_season_features["target_season"] = (
    player_season_features["season"].map(subsequent_season)
)

model_data = (
    player_season_features.dropna(subset=["target_season"])
    .merge(
        early_fixture_targets,
        on=["target_season", "name"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(["season", "name"])
    .reset_index(drop=True)
)

assert not model_data.duplicated(["season", "name"]).any()
print(model_data.groupby(["season", "target_season"]).size())
model_data.head()

## Player-level train/validation split


In [ ]:
model_features = CATEGORICAL_COLUMNS + [
    f"season_sum_{feature}" for feature in NUMERIC_FEATURES
]

train_index, valid_index = train_test_split(
    model_data.index, test_size=0.20, random_state=RANDOM_STATE
)
X_train = model_data.loc[train_index, model_features].copy()
X_valid = model_data.loc[valid_index, model_features].copy()
y_train = model_data.loc[train_index, TARGET_COLUMN].copy()
y_valid = model_data.loc[valid_index, TARGET_COLUMN].copy()
for frame in (X_train, X_valid):
    frame[CATEGORICAL_COLUMNS] = (
        frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    )

print(f"Train players: {len(X_train):,}; validation players: {len(X_valid):,}")


## Unweighted CatBoost model and dummy baseline


In [ ]:
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
validation_predictions["Dummy"] = dummy_model.predict(X_valid)

preseason_model = CatBoostRegressor(
    iterations=163,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
preseason_model.fit(
    X_train,
    y_train,
    cat_features=CATEGORICAL_COLUMNS,
)
validation_predictions["Unweighted CatBoost"] = preseason_model.predict(X_valid)


## Comparison table


In [ ]:
def evaluate_predictions(predictions, top_fraction=0.10):
    evaluation = pd.DataFrame({"actual": y_valid, "predicted": predictions})
    n = max(1, int(np.ceil(len(evaluation) * top_fraction)))
    predicted_top = evaluation.nlargest(n, "predicted")
    actual_top_indices = set(evaluation.nlargest(n, "actual").index)
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        "top_decile_avg_actual_points": predicted_top["actual"].mean(),
        "top_decile_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
        "top_decile_oracle_regret": (
            evaluation.nlargest(n, "actual")["actual"].mean()
            - predicted_top["actual"].mean()
        ),
    }

comparison_table = (
    pd.DataFrame.from_dict(
        {name: evaluate_predictions(preds) for name, preds in validation_predictions.items()},
        orient="index",
    )
    .rename_axis("model").reset_index()
    .sort_values(["top_decile_avg_actual_points", "MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
comparison_table.round(3)


In [ ]:
save_trained_catboost_model(
    model=preseason_model,
    feature_columns=model_features,
    categorical_columns=CATEGORICAL_COLUMNS,
    model_name="Unweighted CatBoost pre-season model",
    model_version="1.0",
    save_path=PRE_SEASON_ARTIFACT_PATH,
)
print(f"Saved pre-season model to {PRE_SEASON_ARTIFACT_PATH}")
